# Классификация CC50 > медиана



In [ ]:
# Установка библиотек для Google Colab
!pip -q install xgboost lightgbm openpyxl seaborn


In [1]:
# Загрузка исходного файла с данными

import os
from google.colab import files

DATA_PATH = '/content/drug_data.xlsx'

if not os.path.exists(DATA_PATH):
    uploaded = files.upload()
    if 'drug_data.xlsx' not in uploaded:

        first_file = next(iter(uploaded.keys()))
        os.rename(first_file, DATA_PATH)

os.makedirs('reports', exist_ok=True)
os.makedirs('models', exist_ok=True)
print(f'Файл данных: {DATA_PATH}')
print('Папки reports/ и models/ готовы')


Файл данных: /content/drug_data.xlsx
Папки reports/ и models/ готовы


## Основной код

In [4]:
"""
Модели классификации для CC50 > медиана
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # Import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

class CC50ClassificationPredictor:
    def __init__(self):
        self.models = {}
        self.results = {}
        self.threshold = None

    def load_and_prepare_data(self):
        print("="*60)
        print("КЛАССИФИКАЦИЯ: CC50 > МЕДИАНА")
        print("="*60)

        df = pd.read_excel(DATA_PATH)
        df = df.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})

        if 'Unnamed: 0' in df.columns:
            df = df.drop('Unnamed: 0', axis=1)

        self.threshold = df['CC50'].median()
        print(f"Медиана CC50: {self.threshold:.3f}")

        y = (df['CC50'] > self.threshold).astype(int)
        print(f"Класс 0: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
        print(f"Класс 1: {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

        feature_cols = [col for col in df.columns if col not in ['IC50', 'CC50', 'SI']]
        X = df[feature_cols].copy()

        # Очистка данных
        constant_features = X.columns[X.nunique() <= 1]
        if len(constant_features) > 0:
            X = X.drop(constant_features, axis=1)

        corr_matrix = X.corr().abs()
        upper_triangle = np.triu(np.ones_like(corr_matrix), k=1)
        high_corr_pairs = np.where((corr_matrix > 0.95) & upper_triangle)

        features_to_remove = set()
        for i in range(len(high_corr_pairs[0])):
            idx1, idx2 = high_corr_pairs[0][i], high_corr_pairs[1][i]
            feature1, feature2 = X.columns[idx1], X.columns[idx2]
            corr1 = abs(np.corrcoef(X[feature1], y)[0,1])
            corr2 = abs(np.corrcoef(X[feature2], y)[0,1])
            if corr1 < corr2:
                features_to_remove.add(feature1)
            else:
                features_to_remove.add(feature2)

        if features_to_remove:
            X = X.drop(list(features_to_remove), axis=1)

        # Replace infinite values with NaN before imputation
        X.replace([np.inf, -np.inf], np.nan, inplace=True)

        # Impute missing values after feature selection
        imputer = SimpleImputer(strategy='mean')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

        print(f"Итоговое количество признаков: {X.shape[1]}")

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        return X, y

    def define_models(self):
        self.models = {
            'Logistic Regression': Pipeline([
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(random_state=42, max_iter=1000))
            ]),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            'XGBoost': xgb.XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
            'LightGBM': lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1),
            'SVM': Pipeline([
                ('scaler', StandardScaler()),
                ('model', SVC(random_state=42, probability=True))
            ])
        }

    def evaluate_models(self):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        for name, model in self.models.items():
            scores = cross_val_score(model, self.X_train, self.y_train, cv=cv, scoring='f1')
            self.results[name] = {'f1_mean': scores.mean(), 'f1_std': scores.std()}
            print(f"{name}: F1 = {scores.mean():.3f} ± {scores.std():.3f}")

        return sorted(self.results.items(), key=lambda x: x[1]['f1_mean'], reverse=True)

    def final_evaluation(self, top_models=3):
        final_results = {}

        sorted_models = sorted(self.results.items(), key=lambda x: x[1]['f1_mean'], reverse=True)

        for name, _ in sorted_models[:top_models]:
            model = self.models[name]
            model.fit(self.X_train, self.y_train)

            y_pred = model.predict(self.X_test)
            y_pred_proba = model.predict_proba(self.X_test)[:, 1]

            final_results[name] = {
                'model': model,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'accuracy': accuracy_score(self.y_test, y_pred),
                'f1': f1_score(self.y_test, y_pred),
                'auc': roc_auc_score(self.y_test, y_pred_proba)
            }

            print(f"{name}: ACC={final_results[name]['accuracy']:.3f}, F1={final_results[name]['f1']:.3f}, AUC={final_results[name]['auc']:.3f}")

        best_model_name = max(final_results.keys(), key=lambda x: final_results[x]['f1'])
        return final_results, best_model_name

    def save_results(self, final_results, best_model_name):
        results_summary = []
        for name, metrics in final_results.items():
            results_summary.append({
                'Model': name,
                'Accuracy': metrics['accuracy'],
                'F1_Score': metrics['f1'],
                'ROC_AUC': metrics['auc']
            })

        results_df = pd.DataFrame(results_summary)
        results_df.to_csv('reports/cc50_classification_results.csv', index=False)

        predictions_df = pd.DataFrame({
            'True_Class': self.y_test.values,
            'Predicted_Class': final_results[best_model_name]['y_pred'],
            'Predicted_Probability': final_results[best_model_name]['y_pred_proba'],
            'CC50_Threshold': self.threshold
        })
        predictions_df.to_csv('reports/cc50_classification_predictions.csv', index=False)

def main():
    predictor = CC50ClassificationPredictor()
    X, y = predictor.load_and_prepare_data()
    predictor.define_models()
    predictor.evaluate_models()
    final_results, best_model_name = predictor.final_evaluation()
    predictor.save_results(final_results, best_model_name)

    print(f"\nЛучшая модель: {best_model_name}")
    print(f"F1-Score: {final_results[best_model_name]['f1']:.3f}")

if __name__ == "__main__":
    main()


КЛАССИФИКАЦИЯ: CC50 > МЕДИАНА
Медиана CC50: 411.039
Класс 0: 502 (50.1%)
Класс 1: 499 (49.9%)
Итоговое количество признаков: 161
Logistic Regression: F1 = 0.736 ± 0.019
Random Forest: F1 = 0.744 ± 0.008
XGBoost: F1 = 0.751 ± 0.016
LightGBM: F1 = 0.747 ± 0.015
SVM: F1 = 0.767 ± 0.009
SVM: ACC=0.721, F1=0.731, AUC=0.814
XGBoost: ACC=0.721, F1=0.736, AUC=0.839
LightGBM: ACC=0.751, F1=0.769, AUC=0.855

Лучшая модель: LightGBM
F1-Score: 0.769
